
# Panglial Connexin Dynamics

This notebook focuses on oligodendroglial `GJC2` (Cx47), oligodendroglial `GJB1` (Cx32), and astrocytic `GJA1` (Cx43) in the current dataset.

`Celltypes` is used to define oligodendroglial and astrocytic subsets. If the loaded dataset contains a real time column, the notebook uses it. If not, it falls back to lesion class as the main progression axis and shows donor-level variation with `Sample`.


In [12]:

from pathlib import Path
import os
import sys

CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next((path for path in CANDIDATES if (path / "src").exists() and (path / "config").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repository root. Start Jupyter from the repo root or notebooks directory.")

MPL_CACHE_DIR = REPO_ROOT / ".cache" / "matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from cx47_oligo.exports import ensure_results_dir, save_current_figure, save_json, save_table
from cx47_oligo.h5ad_tools import adata_overview, expression_frame, load_h5ad, obs_column_summary
from cx47_oligo.scanpy_tools import grouped_gene_expression_stats, obs_keyword_mask

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#fbfcfd",
        "axes.edgecolor": "#d7dde2",
        "grid.color": "#dde4ea",
        "grid.alpha": 0.75,
    }
)


In [13]:

DATA_ROOT = REPO_ROOT / "data" / "raw"
LOCAL_H5AD_FILES = sorted(DATA_ROOT.glob("*.h5ad"))

DATASET_PATH = DATA_ROOT / "jakel_et_al.h5ad"
if not DATASET_PATH.exists():
    DATASET_PATH = LOCAL_H5AD_FILES[0] if LOCAL_H5AD_FILES else None
if DATASET_PATH is None or not DATASET_PATH.exists():
    raise FileNotFoundError("No .h5ad file found under data/raw. Add one or set DATASET_PATH manually.")

RESULTS_DIR = ensure_results_dir(REPO_ROOT, "01_panglial_network", DATASET_PATH)
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET_PATH)
print("Results:", RESULTS_DIR)

save_json(
    {
        "dataset_path": str(DATASET_PATH),
        "results_dir": str(RESULTS_DIR),
        "celltype_column": "Celltypes",
        "target_genes": ["GJC2", "GJB1", "GJA1"],
    },
    RESULTS_DIR / "run_metadata.json",
)


Dataset: /Users/chrislangseth/work/karolinska_institutet/projects/cx47-oligo/data/raw/jakel_et_al.h5ad
Results: /Users/chrislangseth/work/karolinska_institutet/projects/cx47-oligo/results/01_panglial_network/jakel_et_al


PosixPath('/Users/chrislangseth/work/karolinska_institutet/projects/cx47-oligo/results/01_panglial_network/jakel_et_al/run_metadata.json')

In [14]:

adata = load_h5ad(DATASET_PATH)
display(adata_overview(adata))

metadata_columns = [column for column in ["Sample", "Condition", "Lesion", "Celltypes"] if column in adata.obs.columns]
metadata_summary = obs_column_summary(adata).loc[lambda df: df["column"].isin(metadata_columns)]
display(metadata_summary)
save_table(metadata_summary, RESULTS_DIR / "core_obs_summary.csv", index=False)

for column in metadata_columns:
    values = sorted(map(str, adata.obs[column].dropna().unique()))
    print(f"{column}: {values}")

connexin_frame = expression_frame(adata, ["GJC2", "GJB1", "GJA1"])
print("Detected connexin genes:", list(connexin_frame.columns))


,metric,value
0,n_obs,17799
1,n_vars,21581
2,obs_columns,11
3,var_columns,2
4,layers,(none)
5,obsm,(none)
6,uns_keys,"source_dataset, source_stats"
7,raw_present,False


,column,dtype,n_unique,null_fraction,examples
0,Condition,category,2,0.0,"Ctrl, MS"
1,Lesion,category,6,0.0,"Ctrl, NAWM, RM, CA"
3,Sample,category,20,0.0,"CO28, CO25, MS176_NAWM, MS176_RM"
4,Celltypes,category,23,0.0,"COPs, Neuron1, Neuron2, Neuron3"


Sample: ['CO14', 'CO25', 'CO28', 'CO39', 'MS121_A2', 'MS121_A3', 'MS121_CA', 'MS121_NAWM', 'MS122_A', 'MS122_CI', 'MS122_NAWM', 'MS176_CA', 'MS176_CI', 'MS176_NAWM', 'MS176_RM', 'MS242_CA2', 'MS242_CA5', 'MS242_CI', 'MS242_RM', 'SD48/16']
Condition: ['Ctrl', 'MS']
Lesion: ['A', 'CA', 'CI', 'Ctrl', 'NAWM', 'RM']
Celltypes: ['Astrocytes', 'Astrocytes2', 'COPs', 'Endothelial_cells1', 'Endothelial_cells2', 'ImOlGs', 'Immune_cells', 'Macrophages', 'Microglia_Macrophages', 'Neuron1', 'Neuron2', 'Neuron3', 'Neuron4', 'Neuron5', 'OPCs', 'Oligo1', 'Oligo2', 'Oligo3', 'Oligo4', 'Oligo5', 'Oligo6', 'Pericytes', 'Vasc_smooth_muscle']
Detected connexin genes: ['GJC2', 'GJB1', 'GJA1']


In [15]:

CELLTYPE_COLUMN = "Celltypes"
LESION_COLUMN = "Lesion" if "Lesion" in adata.obs.columns else None
SAMPLE_COLUMN = next((column for column in ["Sample", "sample", "donor", "patient", "case"] if column in adata.obs.columns), None)
TIME_COLUMN = next((column for column in ["Timepoint", "timepoint", "Time", "time", "Day", "day", "Week", "week", "dpi", "DPI"] if column in adata.obs.columns), None)

LESION_ORDER = []
if LESION_COLUMN is not None:
    lesion_values = set(map(str, adata.obs[LESION_COLUMN].dropna().unique()))
    LESION_ORDER = [label for label in ["Ctrl", "NAWM", "A", "CA", "CI", "RM"] if label in lesion_values]
    if not LESION_ORDER:
        LESION_ORDER = sorted(lesion_values)
    adata.obs["Lesion_ordered"] = pd.Categorical(
        adata.obs[LESION_COLUMN].astype(str),
        categories=LESION_ORDER,
        ordered=True,
    )

oligo_mask = obs_keyword_mask(adata, CELLTYPE_COLUMN, ["oligo", "opc", "cop", "imolg"])
astro_mask = obs_keyword_mask(adata, CELLTYPE_COLUMN, ["astro"])

adata_oligo = adata[oligo_mask].copy()
adata_astro = adata[astro_mask].copy()

if TIME_COLUMN is None:
    display(
        Markdown(
            "**Time column:** none found in the current dataset. "
            "The plots below therefore use `Lesion` as the main progression axis and `Sample` for donor-level variation."
        )
    )
else:
    display(Markdown(f"**Time column:** `{TIME_COLUMN}`"))

print("Oligodendroglial cells:", adata_oligo.n_obs)
display(adata.obs.loc[oligo_mask, CELLTYPE_COLUMN].value_counts().rename("n_cells").to_frame())
print("Astrocytes:", adata_astro.n_obs)
display(adata.obs.loc[astro_mask, CELLTYPE_COLUMN].value_counts().rename("n_cells").to_frame())

def reorder_frame(frame, categories):
    if frame.empty or not categories:
        return frame
    ordered = [category for category in categories if category in frame.index]
    remainder = [label for label in frame.index if label not in ordered]
    return frame.reindex(ordered + remainder)

def plot_heatmap(frame, title, cmap="mako"):
    fig, ax = plt.subplots(figsize=(max(6, 1.1 * len(frame.columns) + 2), max(4, 0.45 * len(frame.index) + 2)))
    sns.heatmap(frame, cmap=cmap, linewidths=0.4, linecolor="white", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout()

def subset_expression_frame(adata_subset, genes):
    expr = expression_frame(adata_subset, genes)
    meta_columns = [column for column in [TIME_COLUMN, LESION_COLUMN, SAMPLE_COLUMN, CELLTYPE_COLUMN] if column is not None and column in adata_subset.obs.columns]
    frame = adata_subset.obs[meta_columns].copy()
    if "Lesion_ordered" in adata_subset.obs.columns:
        frame["Lesion_ordered"] = adata_subset.obs["Lesion_ordered"]
    return frame.join(expr)

def plot_sample_means(sample_means, group_col, genes, title):
    long_df = sample_means.melt(
        id_vars=[group_col, SAMPLE_COLUMN],
        value_vars=genes,
        var_name="gene",
        value_name="expression",
    )
    if group_col == "Lesion_ordered" and LESION_ORDER:
        long_df[group_col] = pd.Categorical(long_df[group_col], categories=LESION_ORDER, ordered=True)

    fig, ax = plt.subplots(figsize=(max(8, 1.2 * long_df[group_col].nunique()), 5.5))
    if len(genes) == 1:
        sns.boxplot(data=long_df, x=group_col, y="expression", ax=ax, showfliers=False, color="#17465f")
        sns.stripplot(data=long_df, x=group_col, y="expression", ax=ax, color="#d88c32", alpha=0.8, size=6)
    else:
        sns.boxplot(data=long_df, x=group_col, y="expression", hue="gene", ax=ax, showfliers=False)
        sns.stripplot(
            data=long_df,
            x=group_col,
            y="expression",
            hue="gene",
            ax=ax,
            dodge=True,
            alpha=0.7,
            linewidth=0.5,
            edgecolor="white",
        )
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles[: len(genes)], labels[: len(genes)], title="Gene", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    ax.set_title(title)
    ax.set_xlabel(group_col.replace("_", " "))
    ax.set_ylabel("Mean expression per sample")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    return long_df


**Time column:** none found in the current dataset. The plots below therefore use `Lesion` as the main progression axis and `Sample` for donor-level variation.

Oligodendroglial cells: 8774


,n_cells
Celltypes,
Oligo2,1839
Oligo4,1579
Oligo6,1484
Oligo5,1167
Oligo1,1129
Oligo3,775
OPCs,352
COPs,242
ImOlGs,207


Astrocytes: 1242


,n_cells
Celltypes,
Astrocytes,1046
Astrocytes2,196
Neuron4,0
Pericytes,0
Oligo6,0
Oligo5,0
Oligo4,0
Oligo3,0
Oligo2,0



## Lesion-State Summaries

These cells quantify mean expression and the fraction of expressing cells across lesion classes.


In [16]:

OLIGO_GENES = ["GJC2", "GJB1"]
OLIGO_GROUP = "Lesion_ordered" if LESION_COLUMN is not None else CELLTYPE_COLUMN

oligo_stats = grouped_gene_expression_stats(adata_oligo, groupby=OLIGO_GROUP, genes=OLIGO_GENES)
oligo_n = reorder_frame(oligo_stats["n_cells"], LESION_ORDER)
oligo_mean = reorder_frame(oligo_stats["mean_expression"], LESION_ORDER)
oligo_pct = reorder_frame(oligo_stats["pct_expressing"], LESION_ORDER)

display(oligo_n)
display(oligo_mean.round(3))
display(oligo_pct.round(1))

save_table(oligo_n, RESULTS_DIR / "oligo_cell_counts_by_lesion.csv")
save_table(oligo_mean, RESULTS_DIR / "oligo_connexin_mean_by_lesion.csv")
save_table(oligo_pct, RESULTS_DIR / "oligo_connexin_pct_expressing_by_lesion.csv")

plot_heatmap(oligo_mean, "Oligodendroglial Cx47 (GJC2) and Cx32 (GJB1) by lesion", cmap="crest")
save_current_figure(FIGURES_DIR / "oligo_connexin_mean_by_lesion.png")

plot_heatmap(oligo_pct, "Percent of oligodendroglia expressing Cx47 / Cx32 by lesion", cmap="rocket")
save_current_figure(FIGURES_DIR / "oligo_connexin_pct_expressing_by_lesion.png")


,n_cells
Lesion_ordered,
Ctrl,4037
NAWM,1576
A,1184
CA,465
CI,1149
RM,363


,GJC2,GJB1
Lesion_ordered,,
Ctrl,0.203,0.403
NAWM,0.270,0.423
A,0.398,0.424
CA,0.410,0.250
CI,0.262,0.405
RM,0.157,0.080


,GJC2,GJB1
Lesion_ordered,,
Ctrl,10.6,20.2
NAWM,11.7,17.6
A,16.1,17.5
CA,15.7,10.3
CI,11.2,16.9
RM,5.8,2.8


PosixPath('/Users/chrislangseth/work/karolinska_institutet/projects/cx47-oligo/results/01_panglial_network/jakel_et_al/figures/oligo_connexin_pct_expressing_by_lesion.png')

In [17]:

ASTRO_GENES = ["GJA1"]
ASTRO_GROUP = "Lesion_ordered" if LESION_COLUMN is not None else CELLTYPE_COLUMN

astro_stats = grouped_gene_expression_stats(adata_astro, groupby=ASTRO_GROUP, genes=ASTRO_GENES)
astro_n = reorder_frame(astro_stats["n_cells"], LESION_ORDER)
astro_mean = reorder_frame(astro_stats["mean_expression"], LESION_ORDER)
astro_pct = reorder_frame(astro_stats["pct_expressing"], LESION_ORDER)

display(astro_n)
display(astro_mean.round(3))
display(astro_pct.round(1))

save_table(astro_n, RESULTS_DIR / "astro_cell_counts_by_lesion.csv")
save_table(astro_mean, RESULTS_DIR / "astro_cx43_mean_by_lesion.csv")
save_table(astro_pct, RESULTS_DIR / "astro_cx43_pct_expressing_by_lesion.csv")

plot_heatmap(astro_mean, "Astrocytic Cx43 (GJA1) by lesion", cmap="mako")
save_current_figure(FIGURES_DIR / "astro_cx43_mean_by_lesion.png")

plot_heatmap(astro_pct, "Percent of astrocytes expressing Cx43 by lesion", cmap="flare")
save_current_figure(FIGURES_DIR / "astro_cx43_pct_expressing_by_lesion.png")


,n_cells
Lesion_ordered,
Ctrl,454
NAWM,244
A,166
CA,232
CI,115
RM,31


,GJA1
Lesion_ordered,
Ctrl,1.475
NAWM,1.779
A,1.961
CA,2.157
CI,1.913
RM,1.730


,GJA1
Lesion_ordered,
Ctrl,55.9
NAWM,68.0
A,66.9
CA,61.2
CI,65.2
RM,48.4


PosixPath('/Users/chrislangseth/work/karolinska_institutet/projects/cx47-oligo/results/01_panglial_network/jakel_et_al/figures/astro_cx43_pct_expressing_by_lesion.png')


## Donor-Level Variation

The next cell aggregates expression within `Sample` and lesion class so you can separate lesion trends from donor heterogeneity.


In [18]:

if LESION_COLUMN is None or SAMPLE_COLUMN is None:
    print("Need both Lesion and Sample metadata for donor-level summaries.")
else:
    oligo_sample_means = (
        subset_expression_frame(adata_oligo, OLIGO_GENES)
        .groupby(["Lesion_ordered", SAMPLE_COLUMN], observed=True)[OLIGO_GENES]
        .mean()
        .reset_index()
        .sort_values(["Lesion_ordered", SAMPLE_COLUMN])
    )
    display(oligo_sample_means)
    save_table(oligo_sample_means, RESULTS_DIR / "oligo_connexin_sample_means_by_lesion.csv", index=False)
    plot_sample_means(oligo_sample_means, "Lesion_ordered", OLIGO_GENES, "Oligodendroglial connexins by lesion and sample")
    save_current_figure(FIGURES_DIR / "oligo_connexin_sample_means_by_lesion.png")

    astro_sample_means = (
        subset_expression_frame(adata_astro, ASTRO_GENES)
        .groupby(["Lesion_ordered", SAMPLE_COLUMN], observed=True)[ASTRO_GENES]
        .mean()
        .reset_index()
        .sort_values(["Lesion_ordered", SAMPLE_COLUMN])
    )
    display(astro_sample_means)
    save_table(astro_sample_means, RESULTS_DIR / "astro_cx43_sample_means_by_lesion.csv", index=False)
    plot_sample_means(astro_sample_means, "Lesion_ordered", ASTRO_GENES, "Astrocytic Cx43 by lesion and sample")
    save_current_figure(FIGURES_DIR / "astro_cx43_sample_means_by_lesion.png")


,Lesion_ordered,Sample,GJC2,GJB1
0,Ctrl,CO14,0.384189,0.526490
1,Ctrl,CO25,0.316615,0.513463
2,Ctrl,CO28,0.138492,0.553712
3,Ctrl,CO39,0.175063,0.374618
4,Ctrl,SD48/16,0.078827,0.259118
5,NAWM,MS121_NAWM,0.664496,0.996805
6,NAWM,MS122_NAWM,0.154001,0.499542
7,NAWM,MS176_NAWM,0.208334,0.180752
8,A,MS121_A2,0.674023,0.547538
9,A,MS121_A3,0.481820,0.328405


,Lesion_ordered,Sample,GJA1
0,Ctrl,CO14,2.454877
1,Ctrl,CO25,2.406706
2,Ctrl,CO28,0.911637
3,Ctrl,CO39,1.336094
4,Ctrl,SD48/16,0.344741
5,NAWM,MS121_NAWM,2.128128
6,NAWM,MS122_NAWM,1.586174
7,NAWM,MS176_NAWM,1.971898
8,A,MS121_A2,1.838673
9,A,MS121_A3,2.182123



## Timepoint Section

This cell only runs if the loaded dataset includes a real timepoint column.


In [19]:

if TIME_COLUMN is None:
    print("No explicit timepoint column is available in the current dataset.")
    print("If you switch DATASET_PATH to a remyelination dataset with day/week metadata, this section will activate without other changes.")
else:
    oligo_time_stats = grouped_gene_expression_stats(adata_oligo, groupby=TIME_COLUMN, genes=OLIGO_GENES)
    astro_time_stats = grouped_gene_expression_stats(adata_astro, groupby=TIME_COLUMN, genes=ASTRO_GENES)

    display(oligo_time_stats["mean_expression"].round(3))
    display(astro_time_stats["mean_expression"].round(3))

    save_table(oligo_time_stats["mean_expression"], RESULTS_DIR / f"oligo_connexin_mean_by_{TIME_COLUMN}.csv")
    save_table(astro_time_stats["mean_expression"], RESULTS_DIR / f"astro_cx43_mean_by_{TIME_COLUMN}.csv")

    if LESION_COLUMN is not None:
        oligo_time_lesion = (
            subset_expression_frame(adata_oligo, OLIGO_GENES)
            .groupby([TIME_COLUMN, LESION_COLUMN], observed=True)[OLIGO_GENES]
            .mean()
        )
        astro_time_lesion = (
            subset_expression_frame(adata_astro, ASTRO_GENES)
            .groupby([TIME_COLUMN, LESION_COLUMN], observed=True)[ASTRO_GENES]
            .mean()
        )
        display(oligo_time_lesion.round(3))
        display(astro_time_lesion.round(3))
        save_table(oligo_time_lesion, RESULTS_DIR / f"oligo_connexin_mean_by_{TIME_COLUMN}_and_lesion.csv")
        save_table(astro_time_lesion, RESULTS_DIR / f"astro_cx43_mean_by_{TIME_COLUMN}_and_lesion.csv")


No explicit timepoint column is available in the current dataset.
If you switch DATASET_PATH to a remyelination dataset with day/week metadata, this section will activate without other changes.



## Next Step

If you want true temporal trajectories rather than lesion-state proxies, point `DATASET_PATH` to a dataset with explicit day, week, or timepoint metadata. The lesion and donor-level cells above can still be reused unchanged.
